# netweave Real-Data Example: Charleston, SC

This notebook demonstrates a complete workflow using **real data**:

1. **Geographic boundary** from `geodatasets` (Charleston MSA census tracts)
2. **Transit network** from Transitland API (GTFS feeds)
3. **Pedestrian network** from OpenStreetMap via `pandarm.Network.from_gdf`
4. **Integration** into a multimodal network for accessibility analysis

## Requirements

- `TRANSITLAND_API_KEY` environment variable (get one free at https://transit.land)
- `geodatasets` for sample boundary data
- `osmnx`, which `pandarm.Network.from_gdf` uses to download OSM data
- `pandarm`, which holds the finished network and runs the analysis

## API Overview

| Step | Function | Input | Output |
|------|----------|-------|--------|
| 1 | `geodatasets.get_path()` | dataset name | GeoPackage path |
| 2 | `gpd.read_file()` | path | GeoDataFrame |
| 3 | `search_feeds(polygon=gdf)` | GeoDataFrame | Feed metadata |
| 4 | `fetch_feeds()` | feed keys | GTFSFeed |
| 5 | `create_transit_network()` | GTFSFeed | TransitNetwork |
| 6 | `load_network_from_bbox()` | bbox | OSMNetwork |
| 7 | `create_osm_network()` | OSMNetwork | weighted OSMNetwork |
| 8 | `integrate()` | TransitNetwork + OSMNetwork or pandarm.Network | IntegratedNetwork |
| 9 | `to_pandarm()` | IntegratedNetwork | pandarm.Network |

In [ ]:
%load_ext autoreload
%autoreload 2


import geodatasets
import geopandas as gpd
import matplotlib.pyplot as plt
import pandarm as pdna

import netweave as nw

# Optional: set your Transitland API key here or as env var
# os.environ["TRANSITLAND_API_KEY"] = "your-key-here"

print("Imports successful!")

## 1. Load Geographic Boundary

We'll use the `geoda charleston1` dataset from `geodatasets` — a GeoPackage of census tracts for the Charleston, SC Metropolitan Statistical Area. This gives us a realistic boundary for querying transit feeds and OSM data.

In [ ]:
# Load Charleston MSA census tracts
path = geodatasets.get_path("geoda charleston1")
charleston_gdf = gpd.read_file(path)

print(f"Loaded {len(charleston_gdf)} census tracts")
print(f"CRS: {charleston_gdf.crs}")
print(f"Total bounds: {charleston_gdf.total_bounds}")

# Plot the boundary
fig, ax = plt.subplots(figsize=(8, 8))
charleston_gdf.plot(ax=ax, edgecolor="black", facecolor="lightblue", alpha=0.5)
ax.set_title("Charleston, SC Metropolitan Statistical Area")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 2. Search for GTFS Feeds on Transitland

Using the Charleston boundary, we query Transitland for GTFS feeds that serve this area. The `search_feeds()` function accepts a GeoDataFrame directly and computes the bounding box automatically.

**Note:** You need a free Transitland API key from https://transit.land. Set it as the `TRANSITLAND_API_KEY` environment variable or pass it directly.

In [ ]:
# Search for GTFS feeds serving the Charleston area
# This requires a Transitland API key
feeds_df = nw.search_feeds(
    polygon=charleston_gdf, limit=10, api_key="NpQXyLZi60tJpG8Oz6XACrs9o1LeDcKr"
)

print(f"Found {len(feeds_df)} GTFS feed(s)")
feeds_df[["feed_key", "name", "spec", "license_redistribution"]].head()

## 3. Download GTFS Feeds

Download the discovered feeds and merge them into a single `GTFSFeed`. The `fetch_feeds()` function searches, downloads, validates and merges in one step. Every feed's IDs gain a prefix from its Onestop ID, so IDs from different agencies cannot collide. A feed that cannot be used, or whose licence does not allow redistribution, is left out with a warning and recorded in `feed.missing`.


In [ ]:
# Download and merge all discovered feeds (search + download in one step)

feed = nw.fetch_feeds(
    polygon=charleston_gdf,
    limit=10,
    api_key="NpQXyLZi60tJpG8Oz6XACrs9o1LeDcKr",
    merge=True,
    skip_no_redistribution=False,  # Download all feeds, even if redistribution is not allowed
)

print(f"Merged feed: {len(feed.stops)} stops, {len(feed.routes)} routes, {len(feed.trips)} trips")
print(f"Left out: {feed.missing}")
feed.stops.head()

## 4. Create Transit Network

Build a transit network from the merged feed for Monday during the AM peak (7–10 AM). Each feed uses its busiest Monday, and `metadata` records the date chosen. The network has a node for each stop and for each route at a stop, one ride edge per connection priced at its mean time, and boarding edges priced at half the mean headway.


In [ ]:
# Create transit network for Monday AM peak
transit = nw.create_transit_network(
    feed,
    day="monday",
    timerange=("07:00:00", "10:00:00"),
    prune=charleston_gdf,
)

print(f"Transit network: {len(transit.nodes)} nodes, {len(transit.edges)} edges")
print(transit.metadata["feeds"])
transit.edges.head()

In [ ]:

transit.edges[transit.edges["edge_type"] == "ride"].lb.explore()

## 5. Build the Pedestrian Network

Download the OpenStreetMap pedestrian network for the Charleston area with `pandarm`. Its impedance is length in meters, which `integrate()` converts to walking time.


In [ ]:
ped_net = pdna.Network.from_gdf(charleston_gdf, twoway=False)

## 6. Integrate Transit + Pedestrian Networks

Join each stop to its nearest pedestrian node. `integrate()` returns a `pandarm.Network` whose single impedance, `minutes`, covers walking, waiting and riding. The node and edge tables gain labels such as `node_type` and `edge_type`, and `net.metadata` records what went into the network.


In [ ]:
# Integrate transit + pedestrian networks
net = nw.integrate(
    transit,
    ped_net,
    max_connector_dist=500.0,  # meters
    walk_speed_mph=3.0,
)

print(f"Integrated network: {len(net.nodes_df)} nodes, {len(net.edges_df)} edges")
print(f"Stops beyond walking reach: {net.metadata['stranded_stops']}")

# Show edge breakdown
net.edges_df["edge_type"].value_counts()

## 7. Visualize the Integrated Network

Plot the transit stops, OSM nodes, and connector edges on a map. This gives a sense of how well the transit network connects to the pedestrian street network.

## 8. Query the Network

The result is already a `pandarm.Network`, ready for accessibility analysis (e.g., cumulative accessibility, nearest POIs). Pedestrian nodes keep the IDs they had in `ped_net`, and points given to `get_node_ids` and `set_pois` snap only to pedestrian nodes. Query results cover every node, so filter them on `node_type`.


In [ ]:
print(f"pandarm.Network: {len(net.node_ids)} nodes, {len(net.edges_df)} edges")
print(f"Minutes range: {net.edges_df['minutes'].min():.2f} – {net.edges_df['minutes'].max():.2f}")

In [ ]:
net.edges_df[net.edges_df.geometry.notna()].lb.explore()

In [ ]:
net.edges_df.info()

In [ ]:
net.edges_df.minutes

In [ ]:
# precompute the network for accessibility queries (max distance in minutes)
# this takes 1.5 mins on my m3 max
# (you dont actually need to precompute; getting nodes from one origin is fast even at large scales)

net.precompute(45)

In [ ]:
# Pedestrian nodes reachable within 45 minutes of the center of the region
center = charleston_gdf.dissolve().centroid
origin = net.get_node_ids(center.x, center.y).iloc[0]
reachable = net.nodes_in_range([origin], 45, imp_name="minutes").destination.tolist()
nodes = net.nodes_df.loc[reachable]
nodes[nodes["node_type"] == "pedestrian"].lb.explore()